# Installer asyncio

pip install asyncio

# Importer asyncio

import asyncio


## Concept central du fichier: `asyncio`

`asyncio` sert à gérer des tâches qui **attendent** sans bloquer tout le programme.

Usage typique :

```text
API requests
downloads
waiting for server response
files/network/database operations
several slow tasks at the same time
```

Ce n’est pas principalement pour accélérer du calcul lourd. C’est surtout utile quand ton programme passe du temps à **attendre**.

---

# 1. Tableau des notions principales

| Notion            | Définition courte                                      | À quoi ça sert                                      |
| ----------------- | ------------------------------------------------------ | --------------------------------------------------- |
| `async def`       | Définit une fonction asynchrone                        | Créer une coroutine                                 |
| coroutine         | Objet représentant une tâche async non encore terminée | Préparer une opération qui pourra être attendue     |
| `await`           | Attend le résultat d’une coroutine                     | Exécuter une tâche async et récupérer son résultat  |
| `asyncio.sleep()` | Pause non bloquante                                    | Simuler attente API / serveur                       |
| `create_task()`   | Lance une coroutine en tâche de fond                   | Démarrer plusieurs tâches en parallèle              |
| `gather()`        | Attend plusieurs tâches ensemble                       | Récupérer tous les résultats                        |
| `wait()`          | Attend selon une condition                             | Par exemple: prendre le premier résultat disponible |
| `TaskGroup()`     | Groupe de tâches structuré                             | Gérer plusieurs tâches + erreurs proprement         |
| `wait_for()`      | Limite de temps                                        | Annuler si une tâche prend trop longtemps           |
| `Semaphore()`     | Limite le nombre de tâches simultanées                 | Éviter de lancer 100 requêtes en même temps         |
| `to_thread()`     | Lance une fonction bloquante dans un thread            | Utiliser `requests.get()` sans bloquer asyncio      |
| `Queue()`         | File d’attente async                                   | Faire communiquer producteur / consommateur         |

---

# 2. Différence fondamentale

| Code                     | Comportement                                 |
| ------------------------ | -------------------------------------------- |
| `time.sleep(3)`          | bloque tout                                  |
| `await asyncio.sleep(3)` | attend mais laisse les autres tâches avancer |

Erreur importante dans ton notebook :

```python
time.sleep(3)
```

dans une fonction `async def` casse l’intérêt de l’async. Il faut utiliser :

```python
await asyncio.sleep(3)
```

---

# 3. Modèle mental simple

```text
def normale:
    je commence
    je bloque
    je finis
```

```text
async def:
    je commence
    quand j’attends, je rends la main
    d’autres tâches avancent
    je reprends plus tard
```

---

# 4. Les blocs du fichier, en version condensée

| Bloc                                | Idée                                           | Ce que tu dois retenir                                            |
| ----------------------------------- | ---------------------------------------------- | ----------------------------------------------------------------- |
| fonction normale `telechargement()` | exécution synchrone                            | 2 appels de 1 seconde = environ 2 secondes                        |
| `async def telechargement2()`       | coroutine                                      | appeler la fonction ne suffit pas; il faut `await`                |
| `create_task()`                     | tâches lancées en arrière-plan                 | une tâche démarre même si tu n’attends son résultat que plus tard |
| `gather()`                          | attendre toutes les tâches                     | bon choix quand tu veux tous les résultats                        |
| `wait(... FIRST_COMPLETED)`         | attendre la première tâche terminée            | utile pour prendre la première réponse disponible                 |
| `TaskGroup()`                       | groupe structuré                               | si une tâche échoue, l’erreur est regroupée                       |
| `wait_for()`                        | timeout                                        | coupe une tâche trop lente                                        |
| `Semaphore(5)`                      | maximum 5 tâches en même temps                 | contrôle la charge                                                |
| `to_thread()`                       | rendre une fonction bloquante compatible async | utile avec `requests`, qui n’est pas async                        |
| `Queue()`                           | transmettre des messages entre tâches          | producteur / consommateur                                         |

---

# 5. Les trois formes à maîtriser maintenant

## A. Une seule tâche async

```python
# This block runs one async task and waits for its result.
message = await telechargement2("mistral")
```

Sens :

```text
Lance la tâche.
Attends sa fin.
Récupère le résultat.
```

---

## B. Deux tâches lancées ensemble

```python
# This block launches two tasks immediately, then waits for their results later.
tache1 = asyncio.create_task(telechargement2("mistral"))
tache2 = asyncio.create_task(telechargement2("lumo"))

message1 = await tache1
message2 = await tache2
```

Sens :

```text
Les deux tâches commencent.
Le programme peut faire autre chose.
Puis on récupère les résultats.
```

---

## C. Plusieurs tâches avec `gather`

```python
# This block starts several async operations and waits for all results together.
resultats = await asyncio.gather(
    telechargement2("mistral"),
    telechargement2("lumo")
)
```

Sens :

```text
Lance tout.
Attend tout.
Retourne une liste de résultats.
```

---

# 6. À ne pas confondre

| Tu écris                            | Ce que ça donne                    |
| ----------------------------------- | ---------------------------------- |
| `telechargement2("mistral")`        | coroutine                          |
| `await telechargement2("mistral")`  | résultat final, par exemple `"ok"` |
| `create_task(telechargement2(...))` | tâche lancée                       |
| `await tache`                       | résultat de la tâche               |
| `plt.show`                          | fonction non exécutée              |
| `plt.show()`                        | fonction exécutée                  |

Même logique que tu as déjà vue avec les méthodes : sans `()`, tu regardes l’objet fonction; avec `()`, tu l’exécutes.

---

# 7. Priorité d’apprentissage

| Priorité | À maîtriser                             | Niveau attendu maintenant                       |
| -------: | --------------------------------------- | ----------------------------------------------- |
|        1 | `async def` + `await`                   | comprendre coroutine vs résultat                |
|        2 | `asyncio.sleep()` vs `time.sleep()`     | savoir pourquoi `time.sleep()` bloque           |
|        3 | `create_task()`                         | comprendre lancement en arrière-plan            |
|        4 | `gather()`                              | savoir attendre plusieurs résultats             |
|        5 | `wait_for()`                            | comprendre timeout                              |
|        6 | `Semaphore`, `Queue`, `run_in_executor` | comprendre l’idée, pas encore maîtrise complète |

---

## Définition finale courte

`asyncio` est une bibliothèque Python pour écrire du code capable de gérer plusieurs tâches d’attente en même temps, sans bloquer tout le programme. C’est surtout utile pour les API, les téléchargements, les serveurs, les fichiers et les opérations réseau.


In [3]:
import time

def telechargement(url: str) -> str:
    print(f"Début du téléchargement sur {url}")
    time.sleep(1) # fait une pause de 1 seconde
    print(f"Fin du téléchagement sur {url}")
    return "ok"

debut = time.time()

message = telechargement("mistral API")
print(message)

message2 = telechargement("mistral API")
print(message2)

print(f"Temp d'exécution :{time.time() - debut} s")

Début du téléchargement sur mistral API
Fin du téléchagement sur mistral API
ok
Début du téléchargement sur mistral API
Fin du téléchagement sur mistral API
ok
Temp d'exécution :2.005044460296631 s


L'éxécution précédente de la fonction synchrone bloque le flux d'exécution global.

On peut transformer cette méthode en **coroutine** avec le mot clé **async**

In [4]:
import asyncio

# async 
async def telechargement2(url: str) -> str:
    print(f"Début du téléchargement sur {url}")
    time.sleep(3) # fait une pause de 1 seconde
    print(f"Fin du téléchagement sur {url}")
    return "ok"

async def main():
    message = await telechargement2("mistral API")
    print(message)
    print(type(message))
    print(type(telechargement("")))
    print(type(await telechargement2("")))

await main()
#asyncio.run(main()) # dans script python classique

Début du téléchargement sur mistral API
Fin du téléchagement sur mistral API
ok
<class 'str'>
Début du téléchargement sur 
Fin du téléchagement sur 
<class 'str'>
Début du téléchargement sur 
Fin du téléchagement sur 
<class 'str'>


Créer une tâche exploitant la corouine avec **asyncio.create_task(couroutine_a_appeler()).

create_task => créer la tâche et la lancer
await tache => attend sont résultat

In [15]:
async def telechargement_task(url: str) -> str:
    print(f"Début du téléchargement sur {url}")

    # simule le temp execution / le temp de réponse d'une api
    await asyncio.sleep(3) # fait une pause de 1 seconde

    print(f"Fin du téléchagement sur {url}")
    return "ok"

async def main_task():
    tache1 = asyncio.create_task(telechargement_task("mistral API"))
    tache2 = asyncio.create_task(telechargement_task("Lumo AI"))
    print("une  autre action")
    await asyncio.sleep(1) # comme time.sleep : attend un certains nombre de seondes
    print("autre opération")
    message_tache1 =await tache1 # continue l'exécution de la tache jusqu'à avoir un résultat de ma tache
    print(message_tache1)
    message_tache2 =await tache1 # continue l'exécution de la tache jusqu'à avoir un résultat de ma tache
    print(message_tache2)

await main_task()

une  autre action
Début du téléchargement sur mistral API
Début du téléchargement sur Lumo AI
autre opération
Fin du téléchagement sur mistral API
Fin du téléchagement sur Lumo AI
ok
ok


On peut indiquer à asyncio qu'un ensemble de coroutine doivent toute être terminée avant de continuer l'exécution du code.

Ceci est possible grâce à **asyncio.gather()**

In [6]:
import asyncio
async def meteo ():
    await asyncio.sleep(2)
    return "17°C"
async def actualites():
    await asyncio.sleep(1)
    return ["Python 3.15 est sorti", "Incendie des hautes-fagnes est maîtrisé"]

async def main_gather():
    # attente de la fin de toutes les coroutines dans gather
    resultats = await asyncio.gather(meteo(),actualites())

    resultat_meteo = resultats[0]
    resultat_actus = resultats[1]

    print(f"Météo reçue: il fait {resultat_meteo}")
    print(f"Actualités reçues: {resultat_actus}")

await main_gather()


Météo reçue: il fait 17°C
Actualités reçues: ['Python 3.15 est sorti', 'Incendie des hautes-fagnes est maîtrisé']


Dans certains cas, nos taches vont servir à récupérer la même ressources mais sans la certitude d'avoir une résultat et/ou des vitesse différentes.

Pour ce faire, nous employeront **asyncio.wait()**.

In [7]:
import asyncio
async def actualites_arte():
    await asyncio.sleep(3)
    return ["Python 3.15 est sorti", "Incendie des hautes-fagnes est maîtrisé"]
async def actualites_lesoir():
    await asyncio.sleep(5)
    return ["Python 3.15 est sorti", "Incendie des hautes-fagnes est maîtrisé"]
async def actualites_eX_twitter():
    await asyncio.sleep(10)
    return ["Python 3.15 est sorti", "Incendie des hautes-fagnes est maîtrisé"]

async def main_wait():
    taches = [
        asyncio.create_task(actualites_arte()),
        asyncio.create_task(actualites_lesoir()),
        asyncio.create_task(actualites_eX_twitter())
    ]
    faites, encours = await asyncio.wait(taches, return_when=asyncio.FIRST_COMPLETED)

    tache_complete = faites.pop()
    print(tache_complete.result())

    for tache in encours:
        tache.cancel()

await main_wait()

['Python 3.15 est sorti', 'Incendie des hautes-fagnes est maîtrisé']


In [8]:
import asyncio
async def requete_echouable():
    await asyncio.sleep(2)
    raise Exception("Un probléme avec le server")

async def main_task_group():
    try:
        async with asyncio.TaskGroup() as tg:
            tache1 = tg.create_task(requete_echouable())
            tache2 = tg.create_task(requete_echouable())
    except ExceptionGroup as eg:
        print(f"Une ou plusieurs tâches ont échoué: {eg}")

await main_task_group()

Une ou plusieurs tâches ont échoué: unhandled errors in a TaskGroup (2 sub-exceptions)


 await asyncio.sleep(5) > await asyncio.wait_for(requete_trop_longue(), 3)

In [ ]:
import asyncio

async def requete_trop_longue ():
    await asyncio.sleep(5)
    print("fin de la tâche")

async def main_timeout():
    try:
        # arrêt de la tache + envoie d'une exceptin
        await asyncio.wait_for(requete_trop_longue(), 3)
    except TimeoutError as te:
        print(f"Une tâche a pris trop de temps")

await main_timeout()

fin de la tâche


In [20]:
import asyncio
import random

semaphore = asyncio.Semaphore(5)

async def tache(n):
    async with semaphore:
        print(f"Tâche {n} commence")
        await asyncio.sleep(random.randint(1,5))
        print(f"Tâche {n} se termine")

async def main_semaphore():
    await asyncio.gather(*(tache(i) for i in range(10)))

await main_semaphore()

Tâche 0 commence
Tâche 1 commence
Tâche 2 commence
Tâche 3 commence
Tâche 4 commence
Tâche 0 se termine
Tâche 2 se termine
Tâche 5 commence
Tâche 6 commence
Tâche 1 se termine
Tâche 3 se termine
Tâche 5 se termine
Tâche 6 se termine
Tâche 7 commence
Tâche 8 commence
Tâche 9 commence
Tâche 4 se termine
Tâche 8 se termine
Tâche 9 se termine
Tâche 7 se termine


In [11]:
# Rappel du * pour unpack une collection
list = ["coca", "fanta", "eau plate"]
print(list)
print(*list)

dictionary = dict()
dictionary["falcor"] = "chien"
dictionary["achaiah"] = "chat"

print(dictionary)
print(*dictionary)


['coca', 'fanta', 'eau plate']
coca fanta eau plate
{'falcor': 'chien', 'achaiah': 'chat'}
falcor achaiah


In [12]:
import asyncio
import requests as req

url = "https://my-json-server.typicode.com/RobinPBstorm/my-fake-cooking/recettes/1"

demande_recette = asyncio.to_thread(req.get,url)
resultats = await asyncio.gather(demande_recette, asyncio.sleep(2))
print(resultats)

# to_thead(func, args...) => transforme en coroutine
demande_recette = asyncio.create_task(asyncio.to_thread(req.get,url))
print("une autre demande")
reponse = await demande_recette
print(reponse.content)



[<Response [200]>, None]
une autre demande
b'{\n  "id": 1,\n  "titre": "Ratatouille proven\xc3\xa7ale",\n  "categorieIds": [\n    2,\n    4\n  ],\n  "tempsPreparation": 20,\n  "tempsCuisson": 40,\n  "difficulte": "facile",\n  "portions": 4,\n  "ingredients": [\n    {\n      "nom": "Aubergine",\n      "quantite": 2,\n      "unite": "unit\xc3\xa9"\n    },\n    {\n      "nom": "Courgette",\n      "quantite": 3,\n      "unite": "unit\xc3\xa9"\n    },\n    {\n      "nom": "Poivron rouge",\n      "quantite": 2,\n      "unite": "unit\xc3\xa9"\n    },\n    {\n      "nom": "Tomate",\n      "quantite": 4,\n      "unite": "unit\xc3\xa9"\n    },\n    {\n      "nom": "Oignon",\n      "quantite": 2,\n      "unite": "unit\xc3\xa9"\n    },\n    {\n      "nom": "Ail",\n      "quantite": 3,\n      "unite": "gousse"\n    },\n    {\n      "nom": "Huile d\'olive",\n      "quantite": 4,\n      "unite": "cuill\xc3\xa8re \xc3\xa0 soupe"\n    },\n    {\n      "nom": "Herbes de Provence",\n      "quantite": 1,\

In [13]:
import asyncio
# todo à debugger

def processus_long ():
    str = ''
    r  = 'a' * 1000
    for i in range(20_000):
        str += f"{r}{i}"
    print("fin du processus")


async def main_executor():
    loop = asyncio.get_running_loop()
    # None => TheadPoolExecutor par defaut
    # exécution dans un thread différent
    future = loop.run_in_executor(None, processus_long)

    print("opération 1")
    await asyncio.sleep(5)
    print("opération 2")

    # force l'attente de la fin de mon long processus
    # s'il n'est pas fini 
    await future


await main_executor()

opération 1
opération 2
fin du processus


In [14]:
import asyncio

# file pour les résultats de méthodes asynchrone
# ces resultats seront repris par une autre méthode
queue = asyncio.Queue()

async def emmeteur():
    await asyncio.sleep(1)
    print(f"depuis emmetteur: {queue.qsize()}")
    # put rajoute un élément dans la file
    # s'il n'y a plus de place, il attend qu'une se libére
    await queue.put("Voici un message")
    print(f"depuis emmetteur après émission: {queue.qsize()}")

async def recepteur():
    if queue.qsize() > 0:
        print(f"depuis recepteur: {queue.qsize()}")
        
        # get récupère un élément dans la queue
        # s'il n'y en a pas, il attend qu'un résultat arrive
        message = await queue.get()
        print(message)
        # queue l'item a bien été traité
        queue.task_done()
        print(f"depuis recepteur après reception: {queue.qsize()}")
    else:
        print("recepteur: pas de message")

async def main_queue():
    await emmeteur()
    await recepteur()
    await emmeteur()
    await emmeteur()


await main_queue()    


depuis emmetteur: 0
depuis emmetteur après émission: 1
depuis recepteur: 1
Voici un message
depuis recepteur après reception: 0
depuis emmetteur: 0
depuis emmetteur après émission: 1
depuis emmetteur: 1
depuis emmetteur après émission: 2
